In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings("ignore")

import model as ml_model  # reuses ml/model.py directly, so nothing here can drift from the live app

In [ ]:
# Load the live CSV the web app writes to on every /enroll call
CSV_PATH = "keystroke_data.csv"
df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df)}")
print(f"Unique users: {df['user_id'].nunique()}")
df.head()

In [ ]:
# Pick the user to analyse — paste a real user_id copied from the CSV
TARGET_USER_ID = df["user_id"].iloc[0]
print(f"Analysing user: {TARGET_USER_ID}")

df_pos = df[df["user_id"] == TARGET_USER_ID]
df_neg = df[df["user_id"] != TARGET_USER_ID]

print(f"Positive samples (this user): {len(df_pos)}")
print(f"Negative samples (other users): {len(df_neg)}")

In [ ]:
FEATURE_COLUMNS = ml_model.FEATURE_COLUMNS

X_pos = df_pos[FEATURE_COLUMNS].fillna(0).values

if len(df_neg) < 3:
    neg_rows = ml_model.generate_human_negatives(15, FEATURE_COLUMNS)
    X_neg = pd.DataFrame(neg_rows)[FEATURE_COLUMNS].values
else:
    X_neg = df_neg[FEATURE_COLUMNS].fillna(0).values

X = np.vstack([X_pos, X_neg])
y = np.array([1]*len(X_pos) + [0]*len(X_neg))

print(f"Feature vector size: {X.shape[1]}")
print(f"Class balance — user: {sum(y==1)}, others: {sum(y==0)}")

In [ ]:
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.feature_selection import SelectKBest, f_classif

k = min(len(FEATURE_COLUMNS), X.shape[1])
selector = SelectKBest(f_classif, k=k)
selector.fit(X, y)
selected_features = [FEATURE_COLUMNS[i] for i, m in enumerate(selector.get_support()) if m]
X_sel = X[:, selector.get_support()]

scaler = StandardScaler()
power_transform = PowerTransformer()
X_scaled = scaler.fit_transform(X_sel)
X_transformed = power_transform.fit_transform(X_scaled)

print("Selected features:", selected_features)

In [ ]:
# Grid search — best SVM params
svm_params = {"estimator__C": [0.1, 1, 10], "estimator__gamma": ["scale", "auto", 0.01]}
svm_grid = GridSearchCV(
    BaggingClassifier(estimator=SVC(kernel="rbf", probability=True), n_estimators=5),
    svm_params, cv=min(3, len(X)), scoring="f1"
)
svm_grid.fit(X_transformed, y)
print("Best SVM params:", svm_grid.best_params_)
print("Best SVM F1:", round(svm_grid.best_score_, 3))

In [ ]:
# Grid search — best ANN params
ann_params = {"estimator__hidden_layer_sizes": [(16,), (32,16), (64,32)], "estimator__alpha": [0.0001, 0.001]}
ann_grid = GridSearchCV(
    BaggingClassifier(estimator=MLPClassifier(max_iter=500), n_estimators=5),
    ann_params, cv=min(3, len(X)), scoring="f1"
)
ann_grid.fit(X_transformed, y)
print("Best ANN params:", ann_grid.best_params_)
print("Best ANN F1:", round(ann_grid.best_score_, 3))

In [ ]:
# Grid search — best Random Forest params
rf_params = {"n_estimators": [20, 50, 100], "max_depth": [None, 5, 10]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=min(3, len(X)), scoring="f1")
rf_grid.fit(X_transformed, y)
print("Best RF params:", rf_grid.best_params_)
print("Best RF F1:", round(rf_grid.best_score_, 3))

In [ ]:
# Cross-validated honest metrics (not train-set score — this is what goes on the resume)
best_rf = rf_grid.best_estimator_
y_pred = cross_val_predict(best_rf, X_transformed, y, cv=min(3, len(X)))

print(f"Accuracy: {accuracy_score(y, y_pred):.2%}")
print(f"F1 Score: {f1_score(y, y_pred):.3f}")
print(classification_report(y, y_pred, target_names=["Other user", "This user"]))

In [ ]:
# Train the final model with best-found params and save it —
# the web app picks this up directly since it saves to the same
# trained_models/<user_id>.pkl path the running Flask service reads
ml_model.train_from_csv(TARGET_USER_ID)
print(f"Saved trained_models/{TARGET_USER_ID}.pkl")